# Finding asteroids in Hubble exposures with ROSALIA

***

## What is this? 

`ROSALIA` allows the user to estimate much more than stray-light on Roman WFI exposures. In this notebook, we will demonstrate how to use of ROSALIA to find out which Solar System Objects are in your Roman datasets. Following the previous tutorial, we will use datasets stored in the Roman Nexus cloud. We will access a pre-generated Roman mock observation, generate a ROSALIA exposure object, and then use the `rosalia.core.exposure.get_nearby_ssos()` method to automatically query and propagate the orbits of known Solar System Objects stored in the Minor Planet Center database. 


Additional information about ROSALIA can be found in the [ROSALIA for Roman documentation](https://rosalia.readthedocs.io/en/latest/) article on ReadTheDocs.

## How to start

To run this notebook, please activate the "rosalia" kernel at the top right of your window.

## Imports

- *rosalia* for Roman/WFI stray-light analysis
- *matplotlib* for plotting the resulting asteroids tracks

In [1]:
! pwd

/Users/aborlaff/NASA/ROSALIA/notebooks/HST


In [2]:
%matplotlib inline
import rosalia as rs
import matplotlib.pyplot as plt 

## Quick start

First we select a Roman mock image stored at the S3 bucket in Nexus. We generate a ROSALIA exposure object by using rosalia.core.exposure. ROSALIA will automatically detect that the path is a Nexus S3 file, but it also works with local fits or asdf files. 

In [3]:
# With this line we generate the exposure object. This is the first step in any ROSALIA analysis. 
input_name = "/Users/aborlaff/NASA/ROSALIA_DEPOT/MAST_2026-10-19T1427/MAST_2026-10-19T1427/HST/jfk406olq_flc.fits"
rosalia_exposure = rs.core.exposure(filename=input_name)

DATA / WCS mode


KeyError: 'conversion_megajanskys'

In [41]:
import numpy as np
import astropy.units as u
import astropy.constants as astropy_constants

# fnu = lambda**2 / c * flambda

pixscale = 0.05*u.arcsec
photflam = 1.0094804E-19*u.erg*u.cm**-2/u.AA
photplam = 7.6934736E+03 * u.AA
exptime = 476.00000 * u.s
es = 80 / exptime
flambda = es * photflam  
print(flambda)
print(astropy_constants.c)
fnu = (photplam**2/astropy_constants.c * flambda).to("MJy")

print(fnu/es/(pixscale**2).to("sr")) 

print((photplam**2/astropy_constants.c*photflam).to("MJy")/(pixscale**2).to("sr")) 

mu = -2.5*np.log10(fnu/u.Jy/0.05**2) + 8.9
print(mu)



1.6966057142857146e-20 erg / (Angstrom s cm2)
  Name   = Speed of light in vacuum
  Value  = 299792458.0
  Uncertainty  = 0.0
  Unit  = m / s
  Reference = CODATA 2022
3.391816758090828 MJy s / sr


UnitConversionError: 'Angstrom erg s / (m cm2)' and 'MJy' (spectral flux density) are not convertible

In [32]:
fnu

<Quantity 3.34969418e-08 Jy>

In [ ]:
rosalia_exposure.EXPTIME

The `rosalia_exposure` object contains all the necessary information about this particular exposure. It takes a few seconds to run, but it will provide a lot of info in an easy way.

To estimate the locations of the SSOs, ROSALIA uses another package built by our team named `EPHESSOS` (https://pypi.org/project/ephessos/). `EPHESSOS` automatically queries orbital parameters from the Minor Planet Center database (https://www.minorplanetcenter.net/iau/mpc.html), and propagates the orbits through JPL/Horizons, returning the expected trajectory along the duration of the Roman / WFI exposure. All this happens in the background after using the `rosalia_exposure.get_nearby_ssos()` method. 

In [ ]:
# Here we execute the Solar System Object query method to estimate the trajectories of known SSOs. 
cone_search, ssos_db = rosalia_exposure.get_nearby_ssos(time_step="1m", verbose=False)

The ssos_db is a pandas.DataFrame table that contains the predicted location of all known Solar System Objects in the Minor Planet Center database nearby the focal plane array of the designated Roman WFI exposure.

### Now let's plot the results! 

In [ ]:
plt.style.use('dark_background')

ax = rosalia_exposure.plot_footprint(figsize=(8,5))

for i in range(len(ssos_db)):
    ax.plot(ssos_db[i]["RA_deg_ICRF"], 
            ssos_db[i]["DEC_deg_ICRF"], 
            label=ssos_db[i]["Designation"].iloc[0], 
            linewidth=3)
    ax.legend(frameon=False, loc='center left', bbox_to_anchor=(1, 0.5), ncol=3)
ax.xaxis.set_inverted(True) 
plt.savefig("r0003201001001001004_0004_wfi_ssos.png", dpi=300)
plt.show()

Two known asteroids were detected in this 66 second exposure of WFI, one landing on SCA 12 (2017 SL226) and the other one on SCA 9 (2005 SO132). By tagging them automatically, ROSALIA helps to use unintended observations of SSOs to refine the orbits of asteroids for the Minor Planet Center, and at the same time, separates known objects from potentially new SSOs of interest, enhancing the science output of the observatory. 


This method is compatible with Hubble Space Telescope. Here is one example of use on a ACS/WFC exposure. 

In [ ]:
filename = "./MAST_2026-05-08T2317/HST/j90za9qmq/j90za9qmq_flc.fits"
hst_exp = rs.core.exposure(filename=filename)

cone_search, ephessos_df = hst_exp.get_nearby_ssos(time_step="1m")

from tqdm import tqdm

ax = hst_exp.plot_footprint()
for i in tqdm(range(len(ephessos_df))):
    ax.plot(ephessos_df[i]["RA_deg_ICRF"], ephessos_df[i]["DEC_deg_ICRF"], label=ephessos_df[i]["Designation"].iloc[0])
    ax.legend(frameon=False)
ax.xaxis.set_inverted(True) 
plt.savefig("j90za9qmq_flc_ephessos.png", dpi=300)
plt.show()


## About this Notebook

**Author:** Alejandro S. Borlaff - NASA Ames Research Center (Code S) - a.s.borlaff@nasa.gov 

**Updated on** September 18, 2026. 

<table width="100%" style="border:none; border-collapse:collapse;">

  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>